In [0]:
#Bring in the files
#Completed with the help of Gemini and Databricks Assistant
from pyspark.sql import functions as F

volume_path = "/Volumes/cs483eviction/raw/milwaukee_non_foreclosures/"
target_table = "cs483eviction.cleaned.Milwaukee_cleaned"
#Loop through to create the SQL tables

# Loop through to create the SQL tables
for file in dbutils.fs.ls(volume_path):
    if file.name.endswith(".csv"):
        print(f"Processing: {file.name}")
        
        # 1. Read the CSV
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(file.path)
        )
        
        # 2. Clean column names (Delta/SQL tables don't allow spaces or special chars in headers)
        clean_cols = [
            c.strip().lower().replace(" ", "_").replace("-", "_").replace("/", "_")
            for c in df.columns
        ]
        df = df.toDF(*clean_cols)
        
        # 3. Create a valid SQL table name from the filename
        table_name = ''.join(c if c.isalnum() else '_' for c in file.name.replace(".csv", "")).lower()
        
        # 4. Save as a SQL table in the 'raw' schema
        df.write.mode("overwrite").saveAsTable(f"cs483eviction.raw.{table_name}")
        print(f" -> Saved table: cs483eviction.raw.{table_name}")

In [0]:
%sql
SELECT * FROM cs483eviction.raw.milwaukee_2010

In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.raw.milwaukee_2010

In [0]:
%sql
SELECT * FROM cs483eviction.raw.milwaukee_2020

In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.raw.milwaukee_2020

In [0]:
%sql
--lets see what columns we could realistially merge
SELECT * FROM cs483eviction.cleaned.merged_df

In [0]:
%sql
CREATE OR REPLACE TABLE cs483eviction.cleaned.`2010Milwaukee` AS
SELECT * FROM cs483eviction.raw.milwaukee_2010

In [0]:
%sql
CREATE OR REPLACE TABLE cs483eviction.cleaned.`2020Milwaukee` AS
SELECT * FROM cs483eviction.raw.milwaukee_2020